# Hyperparameter Tuning: XGBoost + Model Comparison
**Renewable Energy Production Prediction**

Random Forest was tuned separately and found to produce results essentially identical to the untuned version — so tuning is not repeated here, and the original (untuned) Random Forest results are used in the comparison instead. This notebook tunes XGBoost using GridSearchCV, then compares all three models (Linear Regression baseline, Random Forest, tuned XGBoost) to select a final best-performing model, followed by SHAP interpretability.

## 1. Load the Cleaned Dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib

df = pd.read_csv("../Data/Cleaned/Cleaned_Production_Data.csv")
print(f"Dataset loaded: {len(df):,} rows, {len(df.columns)} columns")
df.head()

Dataset loaded: 51,862 rows, 31 columns


,Date,Start_Hour,End_Hour,Day_of_Year,Production,Temperature_C,Humidity_Percent,Precipitation_mm,WindSpeed_kmh,Source_Wind,...,Month_Name_February,Month_Name_January,Month_Name_July,Month_Name_June,Month_Name_March,Month_Name_May,Month_Name_November,Month_Name_October,Month_Name_September,Rainfall_Flag_Yes
0,2025-11-30,21,22,334,5281,11.8,72,0.0,5.9,1,...,0,0,0,0,0,0,1,0,0,0
1,2025-11-30,18,19,334,3824,13.5,67,0.0,7.2,1,...,0,0,0,0,0,0,1,0,0,0
2,2025-11-30,16,17,334,3824,17.3,50,0.0,5.1,1,...,0,0,0,0,0,0,1,0,0,0
3,2025-11-30,23,0,334,6120,10.4,69,0.0,5.4,1,...,0,0,0,0,0,0,1,0,0,0
4,2025-11-30,6,7,334,4387,8.2,62,0.0,1.6,1,...,0,0,0,0,0,0,1,0,0,0


## 2. Feature Preparation

Same approach as the baseline and Random Forest/XGBoost notebooks: extract `Year` from `Date`, then drop `Date` entirely.

In [2]:
df["Date"] = pd.to_datetime(df["Date"])
df["Year"] = df["Date"].dt.year

df_model = df.drop(columns=["Date"])

for col in df_model.select_dtypes(include='bool').columns:
    df_model[col] = df_model[col].astype(int)

print("Features available:", df_model.columns.tolist())

Features available: ['Start_Hour', 'End_Hour', 'Day_of_Year', 'Production', 'Temperature_C', 'Humidity_Percent', 'Precipitation_mm', 'WindSpeed_kmh', 'Source_Wind', 'Season_Spring', 'Season_Summer', 'Season_Winter', 'Day_Name_Monday', 'Day_Name_Saturday', 'Day_Name_Sunday', 'Day_Name_Thursday', 'Day_Name_Tuesday', 'Day_Name_Wednesday', 'Month_Name_August', 'Month_Name_December', 'Month_Name_February', 'Month_Name_January', 'Month_Name_July', 'Month_Name_June', 'Month_Name_March', 'Month_Name_May', 'Month_Name_November', 'Month_Name_October', 'Month_Name_September', 'Rainfall_Flag_Yes', 'Year']


## 3. Split Features and Target

In [3]:
X = df_model.drop(columns=["Production"])
y = df_model["Production"]

print("Features used:", X.columns.tolist())

Features used: ['Start_Hour', 'End_Hour', 'Day_of_Year', 'Temperature_C', 'Humidity_Percent', 'Precipitation_mm', 'WindSpeed_kmh', 'Source_Wind', 'Season_Spring', 'Season_Summer', 'Season_Winter', 'Day_Name_Monday', 'Day_Name_Saturday', 'Day_Name_Sunday', 'Day_Name_Thursday', 'Day_Name_Tuesday', 'Day_Name_Wednesday', 'Month_Name_August', 'Month_Name_December', 'Month_Name_February', 'Month_Name_January', 'Month_Name_July', 'Month_Name_June', 'Month_Name_March', 'Month_Name_May', 'Month_Name_November', 'Month_Name_October', 'Month_Name_September', 'Rainfall_Flag_Yes', 'Year']


## 4. Train/Test Split

Same split parameters as all other models (`test_size=0.2`, `random_state=42`) for a fair comparison.

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (41489, 30)
Testing set: (10373, 30)


## 5. Hyperparameter Tuning — XGBoost

In [5]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.7, 1.0]
}

grid_search_xgb = GridSearchCV(
    estimator=xgb.XGBRegressor(random_state=42, eval_metric='rmse', n_jobs=-1),
    param_grid=param_grid_xgb,
    cv=3,
    n_jobs=-1,
    verbose=1,
    scoring='neg_mean_squared_error'
)

print("Starting GridSearchCV for XGBoost...")
grid_search_xgb.fit(X_train, y_train)

print(f"Best parameters: {grid_search_xgb.best_params_}")
best_xgb_model = grid_search_xgb.best_estimator_

Starting GridSearchCV for XGBoost...
Fitting 3 folds for each of 54 candidates, totalling 162 fits


C:\Users\BOSS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\joblib\externals\loky\backend\resource_tracker.py:146: UserWarning: resource_tracker: process died unexpectedly, relaunching. Some folders/semaphores might leak.
  warnings.warn(


TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.

Detailed tracebacks of the workers should have been printed to stderr in the executor process if faulthandler was not disabled.

In [ ]:
y_pred_xgb_tuned = best_xgb_model.predict(X_test)

rmse_xgb_tuned = np.sqrt(mean_squared_error(y_test, y_pred_xgb_tuned))
mae_xgb_tuned = mean_absolute_error(y_test, y_pred_xgb_tuned)
r2_xgb_tuned = r2_score(y_test, y_pred_xgb_tuned)

print("Tuned XGBoost Performance:")
print(f"  RMSE: {rmse_xgb_tuned:.2f} MWh")
print(f"  MAE:  {mae_xgb_tuned:.2f} MWh")
print(f"  R2:   {r2_xgb_tuned:.4f}")

### Tuned XGBoost: Actual vs Predicted & Residuals

In [ ]:
os.makedirs('../Visualisations/Models', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred_xgb_tuned, alpha=0.3, color='#8E44AD', s=10)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             color='red', linewidth=2, linestyle='--')
axes[0].set_xlabel('Actual Production')
axes[0].set_ylabel('Predicted Production')
axes[0].set_title('Tuned XGBoost: Actual vs Predicted', fontweight='bold')

residuals_xgb = y_test - y_pred_xgb_tuned
axes[1].scatter(y_pred_xgb_tuned, residuals_xgb, alpha=0.3, color='#F39C12', s=10)
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Production')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Tuned XGBoost: Residual Plot', fontweight='bold')

plt.tight_layout()
plt.savefig('../Visualisations/Models/tuned_xgboost_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

### Save Tuned XGBoost Model and Results

In [ ]:
xgb_results = pd.DataFrame({
    "Model": ["XGBoost (Tuned)"],
    "RMSE": [rmse_xgb_tuned],
    "MAE": [mae_xgb_tuned],
    "R2": [r2_xgb_tuned]
})
xgb_results.to_csv("../Data/ModelResults/tuned_xgboost_results.csv", index=False)

print("Saved: tuned_xgboost_results.csv")

## 6. Model Comparison

Combining results from the Linear Regression baseline, the original (untuned) Random Forest, and tuned XGBoost. Random Forest tuning is excluded here since it produced results essentially identical to the untuned model — the original Random Forest results (from `RandomForest_XGBoost.ipynb`) are used instead. No retraining happens in this step — results are simply loaded back from disk.

In [ ]:
baseline_results = pd.read_csv("../Data/ModelResults/baseline_model_results.csv")
rf_results = pd.read_csv("../Data/ModelResults/advanced_model_results.csv")  # contains untuned Random Forest + XGBoost
rf_results = rf_results[rf_results["Model"] == "Random Forest"]  # keep only Random Forest row
xgb_tuned_results = pd.read_csv("../Data/ModelResults/tuned_xgboost_results.csv")

comparison = pd.concat([baseline_results, rf_results, xgb_tuned_results], ignore_index=True)
comparison = comparison.sort_values(by="R2", ascending=False).reset_index(drop=True)

print(comparison.to_string(index=False))

### Comparison Chart

In [ ]:
plt.figure(figsize=(8, 5))
colors = ['#28B463' if 'Forest' in m else '#8E44AD' if 'XGBoost' in m else '#2E86C1' for m in comparison["Model"]]
plt.bar(comparison["Model"], comparison["R2"], color=colors)
plt.ylabel("R² Score")
plt.title("Model Comparison — R² Score", fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('../Visualisations/Models/model_comparison_r2.png', dpi=150, bbox_inches='tight')
plt.show()

### Identify and Save the Best Model

In [ ]:
best_model_name = comparison.iloc[0]["Model"]
print(f"Best performing model: {best_model_name}")

comparison.to_csv("../Data/ModelResults/final_model_comparison.csv", index=False)
print("Saved: final_model_comparison.csv")

*(Once you run the cells above, replace this note with your actual winning model and numbers — e.g. "Best Model: Tuned XGBoost — achieved the highest R² and lowest RMSE/MAE among all models tested, and is selected as our final production model for the dashboard.")*

## 7. Interpretability — SHAP

Applied only to the winning model, identified above.

In [ ]:
import shap

# Load whichever model won 
best_model = joblib.load("../Data/ModelResults/tuned_xgboost_model.pkl")

explainer = shap.Explainer(best_model, X_train)
shap_values = explainer(X_test)

shap.summary_plot(shap_values, X_test, feature_names=X.columns, show=False)
plt.tight_layout()
plt.savefig('../Visualisations/Models/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

- Random Forest tuning was tested but found to produce essentially the same results as the untuned model, so the original Random Forest results are used going forward
- Tuned XGBoost using GridSearchCV
- Compared all three models (Linear Regression baseline, Random Forest, tuned XGBoost) using RMSE, MAE, and R²
- Identified and saved the best-performing model as our final production model
- Applied SHAP to understand which features drive predictions most strongly

**Next steps:** update the README with these final Part 1 results.